<a href="https://colab.research.google.com/github/DanielTrindade/nota-secreta-a2a/blob/main/nota_secreta_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nota Secreta — A2A / LLM (Colab)

Notebook pronto para rodar o jogo **Nota Secreta** no Google Colab.

O projeto sobe, como subprocessos HTTP:

- um **serviço LLM** centralizado (`llm_service.py`);
- o **Game Master** (`game_master.py`);
- **1 agente estratégico** (`llm_agent.py`) + **5 agentes aleatórios** (`random_agent.py`).

> **Por que rodamos via `!python run_game.py ...` e não via `import`?**
> O `run_game.py` usa `asyncio.run(...)`, que conflita com o event loop já ativo do Jupyter/Colab. Rodar como processo de shell evita esse conflito e mantém o código original intacto.

**Fluxo recomendado:** seções 1 → 2 → 3 → 4 (mock). A seção 5 (modelo real) é opcional e mais pesada.

## 1. Clonar o repositório

O `REPO_URL` já aponta para o repositório. Se quiser usar um fork, troque a URL abaixo.

In [1]:
REPO_URL = "https://github.com/DanielTrindade/nota-secreta-a2a.git"

import os
repo_dir = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo_dir):
    !git clone $REPO_URL
%cd $repo_dir
!ls -la

Cloning into 'nota-secreta-a2a'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 3), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 117.43 KiB | 985.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/nota-secreta-a2a
total 404
drwxr-xr-x 4 root root   4096 Jun  8 15:37 .
drwxr-xr-x 1 root root   4096 Jun  8 15:37 ..
-rw-r--r-- 1 root root  12753 Jun  8 15:37 base_agent.py
-rw-r--r-- 1 root root 263339 Jun  8 15:37 brazilian_songs.csv
-rw-r--r-- 1 root root   4834 Jun  8 15:37 fasta2a.py
-rw-r--r-- 1 root root  23224 Jun  8 15:37 game_master.py
drwxr-xr-x 8 root root   4096 Jun  8 15:37 .git
-rw-r--r-- 1 root root    124 Jun  8 15:37 .gitignore
-rw-r--r-- 1 root root  19538 Jun  8 15:37 llm_agent.py
-rw-r--r-- 1 root root   4642 Jun  8 15:37 llm_service.py
-rw-r--r-- 1 root root   6424 Jun  8 15:37 nota_secreta_colab.ipynb
-rw-r--r-- 1 r

## 2. Instalar dependências (modo mock)

Para validar a arquitetura em modo mock **não** é preciso instalar `llama-cpp-python` (que é pesado e compila). Instalamos só o essencial:

In [2]:
!pip install -q "fastapi>=0.110" "uvicorn>=0.29" "pydantic>=2.6" "aiohttp>=3.9" "pytest>=8.0"

## 3. Rodar os testes

Testes de pontuação (`tests/test_scoring.py`):

In [ ]:
!python -m pytest tests -v

## 4. Partida em modo mock

Roda uma partida completa **sem modelo real** (rápido, ótimo para validar a infraestrutura e o protocolo do agente). O serviço LLM responde com um texto fixo.

Ao final, o caminho do log é mostrado no terminal e salvo em `logs/`.

In [ ]:
!python run_game.py --force-mock

Variante: **6 agentes estratégicos** em modo mock (todos usando `llm_agent.py`):

In [ ]:
!python run_game.py --all-strategic --force-mock

## 5. (Opcional) Partida com modelo real (GGUF)

Esta seção usa um modelo LLM real (`Phi-3.5-mini-instruct`, ~2,4 GB) via `llama-cpp-python`.

**Atenção:**
- A instalação do `llama-cpp-python` **compila** e pode levar alguns minutos.
- O download do modelo (~2,4 GB) também demora.
- Para acelerar a inferência, use um runtime com **GPU** no Colab (`Ambiente de execução → Alterar tipo de ambiente de execução → GPU`).

### 5.1. Instalar o `llama-cpp-python`

In [5]:
# CPU (sempre funciona). Para GPU, veja a célula alternativa abaixo.
!pip install -q "llama-cpp-python>=0.2.70"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# (Alternativa GPU) Instala wheel com suporte a CUDA — use só se o runtime tiver GPU.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir "llama-cpp-python>=0.2.70"

### 5.2. Baixar o modelo GGUF

In [4]:
import os
os.makedirs('models', exist_ok=True)
MODEL_PATH = 'models/phi-3.5-mini-instruct-q4_k_m.gguf'
MODEL_URL = 'https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf'
if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O $MODEL_PATH $MODEL_URL
!ls -lh models/

models/phi-3.5-mini 100%[===================>]   2.23G   220MB/s    in 13s     
total 2.3G
-rw-r--r-- 1 root root 2.3G Jun  8 15:43 phi-3.5-mini-instruct-q4_k_m.gguf


### 5.3. Rodar a partida com o modelo real

`--llm-max-concurrency 1` evita sobrecarregar o modelo local (uma requisição por vez).

In [6]:
!python run_game.py --model models/phi-3.5-mini-instruct-q4_k_m.gguf --llm-max-concurrency 1

[run_game] LLM Service em http://127.0.0.1:9000
[run_game] Game Master em http://127.0.0.1:8000
INFO:     Started server process [8696]
INFO:     Waiting for application startup.
llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:9000 (Press CTRL+C to quit)
INFO:     127.0.0.1:35010 - "GET /health HTTP/1.1" 200 OK
INFO:     Started server process [8715]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:42800 - "GET /health HTTP/1.1" 200 OK
[run_game] Subindo LLMAgent_1 (strategic) em http://127.0.0.1:8001
INFO:     Started server process [8728]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)
INFO:     127.0.0.1:

## 6. Ler o log da última partida

Transforma o log JSON mais recente numa visualização legível (narrador, dicas, cartas jogadas, votos e evolução da pontuação).

In [7]:
import glob, os
logs = sorted(glob.glob('logs/*.json'), key=os.path.getmtime)
if logs:
    ultimo = logs[-1]
    print('Log mais recente:', ultimo)
    !python render_log_readable.py "$ultimo"
else:
    print('Nenhum log encontrado — rode uma partida nas seções 4 ou 5 primeiro.')

Log mais recente: logs/partida_20260608_161009.json
Log: 2026-06-08T16:10:09.980255

Letras
id  type       name           card_id  title                         lyrics                                                                          
--  ---------  -------------  -------  ----------------------------  --------------------------------------------------------------------------------
0   strategic  LLMAgent_1     172      Sá Marina                     Descendo a rua da ladeira Só quem viu, que pode contar Cheirando a flor de lara…
                              143      Minha Namorada                Se você quer ser minha namorada Ah, que linda namorada Você poderia ser Se quis…
                              67       O Canto da Cidade             A cor dessa cidade sou eu O canto dessa cidade é meu A cor dessa cidade sou eu… 
                              201      Fixação                       Seu rosto na tevê parece um milagre Uma perfeição nos mínimos detalhes Eu mudo… 
1   rand